In [25]:
import time
import cloudscraper
import pandas as pd
from bs4 import BeautifulSoup

target_url = "https://mereja.net/amharic/"

# Using cloudscraper to beat the firewall causing the read timeouts
client = cloudscraper.create_scraper()

# Quick connection loop to handle network drops
for run in range(3):
    try:
        print(f"Opening connection... attempt {run + 1}")
        web_page = client.get(target_url, timeout=30)
        web_page.raise_for_status()
        print("Connected successfully.")
        break
    except Exception:
        if run == 2:
            print("Server dropped connection permanently.")
            raise
        print("Lags detected. Retrying shortly...")
        time.sleep(5)

doc = BeautifulSoup(web_page.text, "html.parser")
collected_rows = []

# Scan layout for news items
for block in doc.find_all("article"):
    
    heading = block.find(["h1", "h2", "h3"])
    if not heading:
        continue
        
    anchor = heading.find("a")
    if not anchor:
        continue
        
    headline = anchor.get_text(strip=True)
    hyperlink = anchor.get("href")
    
    # Grab the timestamp info or default to filler text
    timestamp = block.find("time") or block.find("span", class_="entry-date")
    pub_date = timestamp.get_text(strip=True) if timestamp else "N/A"
    
    # Extract the main body copy snippet
    snippet = block.find("div", class_="entry-summary") or block.find("p")
    body_text = snippet.get_text(strip=True) if snippet else ""
    
    collected_rows.append({
        "Title": headline,
        "Link": hyperlink,
        "Date": pub_date,
        "Article": body_text
    })
    
# Output directly to storage file
if collected_rows:
    dataset = pd.DataFrame(collected_rows)
    # sig mapping prevents amharic characters from glitching inside spreadsheets
    dataset.to_csv("mereja_news.csv", index=False, encoding="utf-8-sig")
    print(f"Successfully compiled {len(collected_rows)} rows to mereja_news.csv")
else:
    print("Page parsed but no structured text found.")

Opening connection... attempt 1
Connected successfully.
Successfully compiled 50 rows to mereja_news.csv
